In [ ]:
# this file uses a diagramatic approach to justify the decisions made in data.py where data is cleaned

# fixing metadata
import pandas as pd
import os 
import numpy as np

metadata_file_path = 'data/others/subject_metadata.xls'
metadata_df = pd.read_excel(metadata_file_path)

#Juc010 doesnt exist in raw dataset, fixing Ju study heights (cm to m)
metadata_df = metadata_df.drop(metadata_df[(metadata_df['Study'] == 'Ju') & (metadata_df['Group'] == 'CO') & (metadata_df['Subjnum'] == 10)].index)
metadata_df.loc[metadata_df['Study'] == 'Ju', 'Height (meters)'] /= 100
# change CO/PD to Co/Pt so it matches file names

metadata_df['Group'] = metadata_df['Group'].replace({'PD': 'Pt', 'CO':'Co'})

# checking raw data (walking files)
walking_file_path = 'data/processed'
rows = []
for file in sorted(os.listdir(walking_file_path)):
    if not file.endswith('csv'):
        continue
    walk = pd.read_csv(os.path.join(walking_file_path, file))
    time = walk.iloc[:, 0] #first col
    forces = walk.iloc[:, 1:] #remaining 18 col with forces
    rows.append({
        'file': file,
        'study': file[:2],
        'n_rows': len(walk),
        'duration_s': time.iloc[-1] - time.iloc[0],
        'has_nan': walk.isna().any().any(),
        'has_negative': (forces < 0).any().any(),
        'steady_100hz': np.allclose(np.diff(time), 0.01),
        # leading rows identical to the first row; 1 = no flat start
        'flat_start_rows': (forces != forces.iloc[0]).any(axis=1).to_numpy().argmax(),
    })
quality_df = pd.DataFrame(rows)

# has_nan / has_negative should be 0; steady_100hz should equal the number of walks
quality_df[['has_nan', 'has_negative', 'steady_100hz']].sum()

# duration by study
quality_df.groupby('study')['duration_s'].describe()
quality_df.boxplot(column='duration_s', by='study')
